In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum
from dataclasses import dataclass
import math

In [ ]:
class Direction(Enum):
    DIR_NONE = 0
    UP = 1
    RIGHT = 2
    DOWN = 3
    LEFT = 4
    

DIR_NONE = Direction.DIR_NONE
UP = Direction.UP
RIGHT = Direction.RIGHT
DOWN = Direction.DOWN
LEFT = Direction.LEFT


@dataclass
class TileData:
    num: int
    direction: Direction
    can_take_belt: list[Direction]
    
    def __int__(self):
        return self.num

In [ ]:
class Tile(Enum):
    EMPTY = 0

    BELT_UP = 1
    BELT_RIGHT = 2
    BELT_DOWN = 3
    BELT_LEFT = 4

    BELT_RIGHT_UP = 5
    BELT_DOWN_RIGHT = 6
    BELT_LEFT_DOWN = 7
    BELT_UP_LEFT = 8

    BELT_UP_RIGHT = 9
    BELT_RIGHT_DOWN = 10
    BELT_DOWN_LEFT = 11
    BELT_LEFT_UP = 12

    # EMPTY = TileData(DIR_NONE, can_take_belt=[UP,RIGHT,DOWN,LEFT])
    # BELT_UP = TileData(UP, [DOWN])
    # BELT_RIGHT = TileData(RIGHT, [LEFT])
    # BELT_DOWN = TileData(DOWN, [UP])
    # BELT_LEFT = TileData(LEFT, [RIGHT])

    # BELT_RIGHT_UP = TileData(UP, [RIGHT])
    # BELT_DOWN_RIGHT = TileData(RIGHT, [DOWN])
    # BELT_LEFT_DOWN = TileData(DOWN, [LEFT])
    # BELT_UP_LEFT = TileData(LEFT, [UP])

    # BELT_UP_RIGHT = TileData(RIGHT, [UP])
    # BELT_RIGHT_DOWN = TileData(DOWN, [RIGHT])
    # BELT_DOWN_LEFT = TileData(LEFT, [DOWN])
    # BELT_LEFT_UP = TileData(UP, [LEFT])

    INSERTER_UP = 13
    INSERTER_RIGHT = 14
    INSERTER_DOWN = 15
    INSERTER_LEFT = 16
    
    # ASS_GEAR = 13

    @classmethod
    def inserter_can_drop(self) -> list:
        return range(0,12+1)

    @classmethod
    def can_take_up(self) -> list:
        return [Tile.BELT_DOWN.value,
                Tile.BELT_UP_LEFT.value,
                Tile.BELT_UP_RIGHT.value,
                Tile.EMPTY.value,
                Tile.INSERTER_UP.value,
                Tile.INSERTER_RIGHT.value,
                Tile.INSERTER_DOWN.value,
                Tile.INSERTER_LEFT.value]

    @classmethod
    def can_take_right(self) -> list:
        return [Tile.BELT_LEFT.value,
                Tile.BELT_RIGHT_DOWN.value,
                Tile.BELT_RIGHT_UP.value,
                Tile.EMPTY.value,
                Tile.INSERTER_UP.value,
                Tile.INSERTER_RIGHT.value,
                Tile.INSERTER_DOWN.value,
                Tile.INSERTER_LEFT.value]

    @classmethod
    def can_take_down(self) -> list:
        return [Tile.BELT_UP.value,
                Tile.BELT_DOWN_LEFT.value,
                Tile.BELT_DOWN_RIGHT.value,
                Tile.EMPTY.value,
                Tile.INSERTER_UP.value,
                Tile.INSERTER_RIGHT.value,
                Tile.INSERTER_DOWN.value,
                Tile.INSERTER_LEFT.value]

    @classmethod
    def can_take_left(self) -> list:
        return [Tile.BELT_RIGHT.value,
                Tile.BELT_LEFT_DOWN.value,
                Tile.BELT_LEFT_UP.value,
                Tile.EMPTY.value,
                Tile.INSERTER_UP.value,
                Tile.INSERTER_RIGHT.value,
                Tile.INSERTER_DOWN.value,
                Tile.INSERTER_LEFT.value]
    
    




In [ ]:
def plot_solve(solve):
    fig, ax = plt.subplots(1,solve.size(-1),figsize=(6*5, 6))
    for i, grid in enumerate(solve.permute(2,0,1)):
        ax[i].matshow(grid.detach().exp().numpy(),vmin=0,vmax=1)
        ax[i].set
        ax[i].set_title(Tile(i))
    plt.show()

In [ ]:
BOX_SIZE = 15
INPUT_POS = (14,2)

In [ ]:
def cawr(x):
    return np.maximum( np.cos(x*np.pi/2)**2 + np.cos(x*np.pi*5 % (np.pi/2))/5 , 0.3)
    # return ( np.cos(x*np.pi/2)**2)
    # return np.cos(x*np.pi*5 % (np.pi/2))/5

    # if x<0.8:
    #     return max(0.4, (1 - x /4))
    # else:
    #     return max(0.2, 0.4 - (x-0.8) /0.2)
 
g=list(map(cawr, np.arange(0,1501)/1501))
print(min(g),max(g))
plt.plot(g)
plt.show()

In [ ]:
def eval(solve, verbose=0):
    items_old = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), -5000.0)
    items_old[INPUT_POS][:] = 0.0

    for _ in range(round(BOX_SIZE*1.8)):
        log_total_mass = items_old.max(dim=-1)[0]


        log_pull = torch.full((BOX_SIZE, BOX_SIZE), -5000.0, device=solve.device)
        log_pull[1:, :] = solve[0:-1, :, Tile.INSERTER_UP.value]
        log_pull = log_pull + np.log(0.5)
        

        # log_total_weight = torch.logaddexp(torch.tensor(0.0, device=solve.device), log_pull) # log(1 + pull)
        # log_self_fraction = -log_total_weight # log(1/1+pull) 
        # log_pull_fraction = log_pull - log_total_weight # log(pull/1+pull)
        log_pull_ammount = torch.minimum(log_pull,log_total_mass)
        
        log_pull_fraction = torch.clamp(log_pull_ammount - log_total_mass)
        
        log_self_fraction = torch.log1p(-torch.exp(log_pull_fraction)) # log(1 - pull_fraction)

        mul_self = items_old + solve + log_self_fraction.unsqueeze(-1)

        items = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), -5000.0)
        for i in range(len(Tile)):
            tile_type = Tile(i)

            if tile_type == Tile.EMPTY:
                # idx = [Tile.EMPTY.value]
                # src = mul_self[:, :, i].unsqueeze(-1).repeat(1, 1, len(Tile))
                # items[:, :, :] = torch.logaddexp(items[:, :, idx], src)
                idx = [Tile.EMPTY.value]
                src = mul_self[:, :, i].unsqueeze(-1).repeat(1, 1, len(idx))
                items[:, :, idx] = torch.logaddexp(items[:, :, idx], src)
             
            if tile_type in [Tile.BELT_UP, Tile.BELT_LEFT_UP, Tile.BELT_RIGHT_UP]:
                idx = Tile.can_take_down()
                src = mul_self[1::, :, i].unsqueeze(-1).repeat(1, 1, len(idx))
                items[0:-1, :, idx] = torch.logaddexp(items[0:-1, :, idx], src)
                
            if tile_type in [Tile.BELT_RIGHT, Tile.BELT_DOWN_RIGHT, Tile.BELT_UP_RIGHT]:
                idx = Tile.can_take_left()
                src = mul_self[:, 0:-1, i].unsqueeze(-1).repeat(1, 1, len(idx))
                items[:, 1::, idx] = torch.logaddexp(items[:, 1::, idx], src)
                
            if tile_type in [Tile.BELT_DOWN, Tile.BELT_LEFT_DOWN, Tile.BELT_RIGHT_DOWN]:
                idx = Tile.can_take_up()
                src = mul_self[0:-1, :, i].unsqueeze(-1).repeat(1, 1, len(idx))
                items[1::, :, idx] = torch.logaddexp(items[1::, :, idx], src)
                
            if tile_type in [Tile.BELT_LEFT, Tile.BELT_DOWN_LEFT, Tile.BELT_UP_LEFT]:
                idx = Tile.can_take_right()
                src = mul_self[:, 1::, i].unsqueeze(-1).repeat(1, 1, len(idx))
                items[:, 0:-1, idx] = torch.logaddexp(items[:, 0:-1, idx], src)

            if tile_type == Tile.INSERTER_UP:
                idx = Tile.inserter_can_drop()
                
                src = (log_total_mass[2::, :] + log_pull_fraction[2::, :]).unsqueeze(-1).repeat(1, 1, len(idx))
                items[0:-2, :, idx] = torch.logaddexp(items[0:-2, :, idx], src)

        sum_items = items.max(-1)[0].exp().sum().item()
        # assert sum_items <= 5, f'Sum should be <= 1. ({sum_items})'
        
        items_old = items.clone()

        if verbose>5:
            plt.matshow(items_old.detach().max(-1)[0].exp().numpy(),vmin=0,vmax=1)
            plt.show()
    return items

In [ ]:
def eval_2(solve, verbose=0):
    items_old = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), -5000.0, device=solve.device)
    
    for _ in range(round(BOX_SIZE * 1.8)):
        
        input_mask = torch.zeros_like(items_old, dtype=torch.bool)
        input_mask[INPUT_POS] = True
        
        new_input_val = torch.logaddexp(items_old, torch.tensor(0.0, device=solve.device))
        items_old = torch.where(input_mask, new_input_val, items_old)


        log_total_mass = items_old.max(dim=-1)[0]


        has_items = log_total_mass > -1000.0

        log_pull = torch.full((BOX_SIZE, BOX_SIZE), -5000.0, device=solve.device)
        log_pull[1:, :] = solve[0:-1, :, Tile.INSERTER_UP.value]
        log_pull = log_pull + np.log(0.5)
        
        log_pull_ammount = torch.minimum(log_pull, log_total_mass)
        
        safe_log_total_mass = torch.where(has_items, log_total_mass, torch.tensor(0.0, device=solve.device))
        log_pull_fraction_raw = log_pull_ammount - safe_log_total_mass
        
        log_pull_fraction = torch.where(
            has_items,
            torch.clamp(log_pull_fraction_raw, max=-1e-5),
            torch.tensor(-5000.0, device=solve.device)
        )
        
        log_self_fraction = torch.log1p(-torch.exp(log_pull_fraction))

        mul_self = items_old + solve + log_self_fraction.unsqueeze(-1)

        layer_updates = []
        
        for i in range(len(Tile)):
            tile_type = Tile(i)
            
            current_layer = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), -5000.0, device=solve.device)

            if tile_type == Tile.EMPTY:
                idx = [Tile.EMPTY.value]
                src = mul_self[:, :, i].unsqueeze(-1).repeat(1, 1, len(idx))
                current_layer[:, :, idx] = src
             
            elif tile_type in [Tile.BELT_UP, Tile.BELT_LEFT_UP, Tile.BELT_RIGHT_UP]:
                idx = Tile.can_take_down()
                src = mul_self[1::, :, i].unsqueeze(-1).repeat(1, 1, len(idx))
                current_layer[0:-1, :, idx] = src
                
            elif tile_type in [Tile.BELT_RIGHT, Tile.BELT_DOWN_RIGHT, Tile.BELT_UP_RIGHT]:
                idx = Tile.can_take_left()
                src = mul_self[:, 0:-1, i].unsqueeze(-1).repeat(1, 1, len(idx))
                current_layer[:, 1::, idx] = src
                
            elif tile_type in [Tile.BELT_DOWN, Tile.BELT_LEFT_DOWN, Tile.BELT_RIGHT_DOWN]:
                idx = Tile.can_take_up()
                src = mul_self[0:-1, :, i].unsqueeze(-1).repeat(1, 1, len(idx))
                current_layer[1::, :, idx] = src
                
            elif tile_type in [Tile.BELT_LEFT, Tile.BELT_DOWN_LEFT, Tile.BELT_UP_LEFT]:
                idx = Tile.can_take_right()
                src = mul_self[:, 1::, i].unsqueeze(-1).repeat(1, 1, len(idx))
                current_layer[:, 0:-1, idx] = src

            elif tile_type == Tile.INSERTER_UP:
                idx = Tile.inserter_can_drop()
                src = (log_total_mass[2::, :] + log_pull_fraction[2::, :]).unsqueeze(-1).repeat(1, 1, len(idx))
                current_layer[0:-2, :, idx] = src
                
            layer_updates.append(current_layer)

        items = layer_updates[0]
        for next_layer in layer_updates[1:]:
            items = torch.logaddexp(items, next_layer)

        items_old = items

        if verbose > 5:
            plt.matshow(items_old.detach().max(-1)[0].exp().numpy(), vmin=0, vmax=1)
            plt.show()

    return items

In [ ]:
logits = torch.full((BOX_SIZE,BOX_SIZE,len(Tile)), -5000.0)
active_tiles = list(range(0,13+1))

logits[:,:,active_tiles] = 5 * torch.randn(BOX_SIZE,BOX_SIZE,len(active_tiles))

logits.requires_grad_()
# solve = smax(solve)

MAX_ITER = 151
START_TEMP = 30

opt = torch.optim.Adam([logits], lr=120)


for iter in range(MAX_ITER):
    temp = cawr(iter/MAX_ITER)


    solve = torch.log_softmax(logits/(temp*START_TEMP), dim=-1)
    # plot_solve(solve)
    # with torch.no_grad():
    solve = solve+ 0.01 * temp * torch.randn_like(solve)

    solve = torch.clamp(solve, max=-0.0001)

    if iter%25==0:
        plot_solve(solve)

    items = eval(solve)
    # path_loss = -items[0, 13, :].mean(dim=-1)
    path_loss = -items[0, 13, :].max(dim=-1)[0]
    
    cost_loss = -( torch.logsumexp(solve[:,:,0], (0,1))/BOX_SIZE**2 )**2 * 1500
    
    # cost_loss = -solve[:,:,0].mean() 
    # l2_non_empty = (logits[:, :, 1:] ** 2).mean() * 0.005
    # loss += 3* ((1-solve.sum(dim=-1))**2).mean()


    # print(f"Grad norm: {grad_norm:.4f} | Empty cells: {empty_ratio:.1%}")
    # print(loss.grad_fn)
    
    # if loss.item()<0.05:
    #     break
    loss = path_loss + cost_loss
    loss.backward()
    # print(logits.grad)
    opt.step()

    grad_norm = logits.grad.abs().mean().item()
    empty_ratio = (solve[:, :, 0] > -0.5).float().mean().item()
    print(path_loss.item(),cost_loss.item(),temp, grad_norm, empty_ratio)

    opt.zero_grad()
    

In [ ]:
import os
from PIL import Image
from pathlib import Path
assets = Path('assets').iterdir()
assets = {k.name[0:-4]:Image.open(k) for k in assets}

In [ ]:
for i in assets.values():
    i.close()

In [ ]:
def render_solve(solve: torch.Tensor):
    fig, axes = plt.subplots(BOX_SIZE, BOX_SIZE, figsize=(8, 8),gridspec_kw={"wspace": 0, "hspace": 0})

    for ax, v in zip(axes.ravel(), solve.reshape(BOX_SIZE*BOX_SIZE,-1)):
        tile = Tile(v.argmax().item())
        # print(tile)
        img = assets.get(tile.name.lower())
        if img:
            ax.imshow(img)
        ax.axis("off")
    plt.show()
render_solve(solve)

In [ ]:
eval(solve,verbose=10)